# Multi-Base Metric Montage Candidates

This notebook searches for consecutive shot-length runs that can be explained as integer multiples of any base length from 1 to 90 seconds. It keeps all matching bases for each candidate segment, chooses a canonical best base, and writes independent outputs under `metric_montage_multi_base_results/`.

In [1]:
from __future__ import annotations

import math
import os
import re
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

SHOT_LENGTH_RESULTS_DIR = Path("./shot_length_results")
WIDE_CSV = SHOT_LENGTH_RESULTS_DIR / "shot_lengths_wide.csv"
LONG_CSV = SHOT_LENGTH_RESULTS_DIR / "shot_lengths_long.csv"

OUTPUT_DIR = Path("./metric_montage_multi_base_results")

MIN_CONSECUTIVE_SHOTS = 3
BASE_LENGTH_SECONDS_RANGE = range(1, 91)
TOLERANCE_SECONDS = 0.2

# "maximal_runs" writes the longest uninterrupted run for each base, then merges bases by segment.
# "all_windows" writes every matching sliding window of exactly MIN_CONSECUTIVE_SHOTS, then merges bases by segment.
RECORD_MODE = "maximal_runs"

WRITE_PER_FILM_CSV = True
WRITE_COMBINED_CSV = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wide CSV: {WIDE_CSV.resolve()}")
print(f"Long CSV: {LONG_CSV.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Wide CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/shot_lengths_wide.csv
Long CSV: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/shot_length_results/shot_lengths_long.csv
Output directory: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/metric_montage_multi_base_results


/Users/caoshuangyuan/anaconda3/envs/coreml4/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def decimal_token(value: float | int) -> str:
    text = f"{float(value):g}"
    return text.replace(".", "p").replace("-", "m")


def safe_filename_part(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"[\\/:*?\"<>|]", "_", text)
    return text or "untitled"


def base_range_label() -> str:
    bases = list(BASE_LENGTH_SECONDS_RANGE)
    if not bases:
        raise ValueError("BASE_LENGTH_SECONDS_RANGE cannot be empty")
    if bases == list(range(int(min(bases)), int(max(bases)) + 1)):
        return f"len{decimal_token(min(bases))}to{decimal_token(max(bases))}"
    return "len" + "-".join(decimal_token(base) for base in bases)


def output_suffix() -> str:
    return f"shot{MIN_CONSECUTIVE_SHOTS}_{base_range_label()}_tlr{decimal_token(TOLERANCE_SECONDS)}"


def shot_column_number(column_name: str) -> int:
    match = re.fullmatch(r"shot_(\d+)", column_name)
    if not match:
        raise ValueError(f"Not a shot column: {column_name}")
    return int(match.group(1))


def sorted_shot_columns(wide_df: pd.DataFrame) -> list[str]:
    columns = [column for column in wide_df.columns if re.fullmatch(r"shot_\d+", column)]
    return sorted(columns, key=shot_column_number)


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    if not WIDE_CSV.exists():
        raise FileNotFoundError(f"Missing wide CSV: {WIDE_CSV}")
    if not LONG_CSV.exists():
        raise FileNotFoundError(f"Missing long CSV: {LONG_CSV}")

    wide_df = pd.read_csv(WIDE_CSV)
    long_df = pd.read_csv(LONG_CSV)
    shot_columns = sorted_shot_columns(wide_df)

    required_long_columns = {"film", "shot_index", "duration_sec", "start_sec", "end_sec", "video_path"}
    missing = required_long_columns - set(long_df.columns)
    if missing:
        raise ValueError(f"Long CSV is missing required columns: {sorted(missing)}")
    if "film" not in wide_df.columns:
        raise ValueError("Wide CSV is missing required column: film")
    if not shot_columns:
        raise ValueError("Wide CSV has no shot_#### columns")
    return wide_df, long_df, shot_columns


wide_df, long_df, shot_columns = load_inputs()
BASE_LENGTH_SECONDS_LIST = [float(base) for base in BASE_LENGTH_SECONDS_RANGE]
print(f"Loaded {len(wide_df)} films from wide CSV.")
print(f"Loaded {len(long_df)} shot rows from long CSV.")
print(f"Testing {len(BASE_LENGTH_SECONDS_LIST)} base length(s): {BASE_LENGTH_SECONDS_LIST[:5]} ... {BASE_LENGTH_SECONDS_LIST[-5:]}")
print(f"Output suffix: {output_suffix()}")

Loaded 454 films from wide CSV.
Loaded 193593 shot rows from long CSV.
Testing 90 base length(s): [1.0, 2.0, 3.0, 4.0, 5.0] ... [86.0, 87.0, 88.0, 89.0, 90.0]
Output suffix: shot3_len1to90_tlr0p2


In [3]:
def film_shots_from_wide(row: pd.Series, shot_columns: list[str]) -> pd.DataFrame:
    rows = []
    for column in shot_columns:
        duration = row[column]
        if pd.isna(duration):
            continue
        rows.append({
            "film": row["film"],
            "shot_index": shot_column_number(column),
            "duration_sec": float(duration),
        })
    return pd.DataFrame(rows)


def base_match(duration_sec: float, base_length_sec: float) -> dict:
    if pd.isna(duration_sec) or duration_sec <= 0:
        return {"matches": False}
    multiple = max(1, int(round(duration_sec / base_length_sec)))
    expected_sec = multiple * base_length_sec
    error_sec = duration_sec - expected_sec
    abs_error_sec = abs(error_sec)
    return {
        "matches": abs_error_sec <= TOLERANCE_SECONDS,
        "multiple": multiple,
        "expected_sec": expected_sec,
        "error_sec": error_sec,
        "abs_error_sec": abs_error_sec,
    }


def quote_long_segment(film: str, start_shot: int, end_shot: int, long_df: pd.DataFrame) -> pd.DataFrame:
    segment = long_df[
        (long_df["film"] == film)
        & (long_df["shot_index"] >= start_shot)
        & (long_df["shot_index"] <= end_shot)
    ].copy()
    return segment.sort_values("shot_index")


def matching_rows_for_base(film_shots: pd.DataFrame, base_length_sec: float) -> pd.DataFrame:
    rows = []
    for _, shot in film_shots.sort_values("shot_index").iterrows():
        match = base_match(float(shot["duration_sec"]), base_length_sec)
        rows.append({**shot.to_dict(), "base_length_sec": base_length_sec, **match})
    return pd.DataFrame(rows)

In [4]:
def base_run_record(film: str, segment: pd.DataFrame, base_length_sec: float) -> dict:
    durations = [round(float(value), 3) for value in segment["duration_sec"].tolist()]
    multiples = [int(value) for value in segment["multiple"].tolist()]
    expected = [round(float(value), 3) for value in segment["expected_sec"].tolist()]
    errors = [round(float(value), 3) for value in segment["error_sec"].tolist()]
    abs_errors = [float(value) for value in segment["abs_error_sec"].tolist()]

    return {
        "film": film,
        "start_shot": int(segment["shot_index"].min()),
        "end_shot": int(segment["shot_index"].max()),
        "shot_count": int(len(segment)),
        "base_length_sec": float(base_length_sec),
        "durations_sec": durations,
        "multiples": multiples,
        "expected_sec": expected,
        "errors_sec": errors,
        "max_abs_error_sec": max(abs_errors),
        "mean_abs_error_sec": sum(abs_errors) / len(abs_errors),
        "max_multiple": max(multiples),
        "mean_multiple": sum(multiples) / len(multiples),
    }


def find_maximal_base_runs(film_shots: pd.DataFrame, base_length_sec: float) -> list[dict]:
    matches = matching_rows_for_base(film_shots, base_length_sec)
    runs = []
    current_rows = []
    previous_shot_index = None

    for _, shot in matches.sort_values("shot_index").iterrows():
        shot_index = int(shot["shot_index"])
        contiguous = previous_shot_index is None or shot_index == previous_shot_index + 1
        if bool(shot["matches"]) and contiguous:
            current_rows.append(shot)
        else:
            if len(current_rows) >= MIN_CONSECUTIVE_SHOTS:
                segment = pd.DataFrame(current_rows)
                runs.append(base_run_record(str(segment["film"].iloc[0]), segment, base_length_sec))
            current_rows = [shot] if bool(shot["matches"]) else []
        previous_shot_index = shot_index

    if len(current_rows) >= MIN_CONSECUTIVE_SHOTS:
        segment = pd.DataFrame(current_rows)
        runs.append(base_run_record(str(segment["film"].iloc[0]), segment, base_length_sec))

    return runs


def find_window_base_runs(film_shots: pd.DataFrame, base_length_sec: float) -> list[dict]:
    matches = matching_rows_for_base(film_shots, base_length_sec).sort_values("shot_index").reset_index(drop=True)
    runs = []
    for start in range(0, len(matches) - MIN_CONSECUTIVE_SHOTS + 1):
        window = matches.iloc[start:start + MIN_CONSECUTIVE_SHOTS].copy()
        expected_indices = list(range(int(window["shot_index"].iloc[0]), int(window["shot_index"].iloc[0]) + MIN_CONSECUTIVE_SHOTS))
        actual_indices = [int(value) for value in window["shot_index"].tolist()]
        if actual_indices != expected_indices:
            continue
        if bool(window["matches"].all()):
            runs.append(base_run_record(str(window["film"].iloc[0]), window, base_length_sec))
    return runs

In [5]:
def select_best_base(base_records: list[dict]) -> dict:
    # Prefer the simplest integer sequence, then lower error, then larger base.
    return sorted(
        base_records,
        key=lambda record: (
            record["max_multiple"],
            round(record["mean_abs_error_sec"], 9),
            round(record["max_abs_error_sec"], 9),
            -record["base_length_sec"],
        ),
    )[0]


def list_to_cell(values, decimals: int | None = None) -> str:
    out = []
    for value in values:
        if isinstance(value, float) and decimals is not None:
            out.append(str(round(value, decimals)))
        else:
            out.append(str(value))
    return ";".join(out)


def candidate_priority_score(row: dict) -> float:
    score = 0.0
    # Cap the shot-count reward so long base=1 integer-second runs do not dominate.
    score += min(row["shot_count"], 8) * 10.0
    score += max(0.0, TOLERANCE_SECONDS - row["best_mean_abs_error_sec"]) * 25.0
    score += max(0.0, TOLERANCE_SECONDS - row["best_max_abs_error_sec"]) * 10.0
    score -= max(0, row["best_max_multiple"] - 4) * 2.0
    score -= max(0, row["matching_base_count"] - 3) * 0.5
    if row["duration_total_sec"] > 90:
        score -= min(25.0, (row["duration_total_sec"] - 90) / 6.0)
    if row["low_information_base_flag"]:
        score -= 55.0
    if row["compact_candidate_flag"] and not row["low_information_base_flag"]:
        score += 8.0
    if row["base_ambiguity_flag"]:
        score -= 2.0
    return round(score, 3)


def merge_base_records_for_segment(base_records: list[dict], long_df: pd.DataFrame) -> dict:
    best = select_best_base(base_records)
    matching_bases = sorted(record["base_length_sec"] for record in base_records)
    quoted = quote_long_segment(best["film"], best["start_shot"], best["end_shot"], long_df)

    if quoted.empty:
        start_sec = math.nan
        end_sec = math.nan
        video_path = ""
    else:
        start_sec = float(quoted["start_sec"].min())
        end_sec = float(quoted["end_sec"].max())
        video_path = str(quoted["video_path"].iloc[0])

    row = {
        "film": best["film"],
        "start_shot": best["start_shot"],
        "end_shot": best["end_shot"],
        "shot_count": best["shot_count"],
        "start_sec": round(start_sec, 3) if not math.isnan(start_sec) else math.nan,
        "end_sec": round(end_sec, 3) if not math.isnan(end_sec) else math.nan,
        "duration_total_sec": round(end_sec - start_sec, 3) if not math.isnan(start_sec) and not math.isnan(end_sec) else math.nan,
        "tolerance_sec": TOLERANCE_SECONDS,
        "all_matching_bases": list_to_cell(matching_bases, decimals=3),
        "matching_base_count": len(matching_bases),
        "base_ambiguity_flag": len(matching_bases) > 3,
        "best_base_length_sec": best["base_length_sec"],
        "best_durations_sec": list_to_cell(best["durations_sec"], decimals=3),
        "best_multiples": list_to_cell(best["multiples"]),
        "best_expected_sec": list_to_cell(best["expected_sec"], decimals=3),
        "best_errors_sec": list_to_cell(best["errors_sec"], decimals=3),
        "best_max_abs_error_sec": round(best["max_abs_error_sec"], 3),
        "best_mean_abs_error_sec": round(best["mean_abs_error_sec"], 3),
        "best_max_multiple": int(best["max_multiple"]),
        "best_mean_multiple": round(best["mean_multiple"], 3),
        "compact_candidate_flag": best["max_multiple"] <= 4 and (not math.isnan(start_sec)) and (end_sec - start_sec <= 90),
        "low_information_base_flag": best["base_length_sec"] <= 1.0,
        "video_path": video_path,
    }
    row["priority_score"] = candidate_priority_score(row)
    return row

In [6]:
def find_candidates_for_film(row: pd.Series, shot_columns: list[str], long_df: pd.DataFrame) -> pd.DataFrame:
    film_shots = film_shots_from_wide(row, shot_columns)
    if film_shots.empty:
        return pd.DataFrame()

    segment_records: dict[tuple[str, int, int], list[dict]] = {}
    for base_length_sec in BASE_LENGTH_SECONDS_LIST:
        if RECORD_MODE == "maximal_runs":
            base_records = find_maximal_base_runs(film_shots, base_length_sec)
        elif RECORD_MODE == "all_windows":
            base_records = find_window_base_runs(film_shots, base_length_sec)
        else:
            raise ValueError("RECORD_MODE must be 'maximal_runs' or 'all_windows'")

        for record in base_records:
            key = (record["film"], record["start_shot"], record["end_shot"])
            segment_records.setdefault(key, []).append(record)

    candidates = [merge_base_records_for_segment(records, long_df) for records in segment_records.values()]
    if not candidates:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    return pd.DataFrame(candidates, columns=OUTPUT_COLUMNS)


OUTPUT_COLUMNS = [
    "film", "start_shot", "end_shot", "shot_count", "start_sec", "end_sec",
    "duration_total_sec", "tolerance_sec", "all_matching_bases", "matching_base_count",
    "base_ambiguity_flag", "best_base_length_sec", "best_durations_sec",
    "best_multiples", "best_expected_sec", "best_errors_sec", "best_max_abs_error_sec",
    "best_mean_abs_error_sec", "best_max_multiple", "best_mean_multiple",
    "compact_candidate_flag", "low_information_base_flag", "priority_score", "video_path",
]

In [7]:
all_candidates = []
suffix = output_suffix()

for _, film_row in tqdm(list(wide_df.sort_values("film").iterrows()), desc="Films"):
    film = str(film_row["film"])
    film_candidates = find_candidates_for_film(film_row, shot_columns, long_df)
    if film_candidates.empty:
        film_candidates = pd.DataFrame(columns=OUTPUT_COLUMNS)
    else:
        film_candidates = film_candidates.sort_values(
            ["priority_score", "shot_count", "best_mean_abs_error_sec"],
            ascending=[False, False, True],
            kind="stable",
        ).reset_index(drop=True)
        all_candidates.append(film_candidates)

    if WRITE_PER_FILM_CSV:
        output_path = OUTPUT_DIR / f"{safe_filename_part(film)}_{suffix}.csv"
        film_candidates.to_csv(output_path, index=False, encoding="utf-8-sig")

if all_candidates:
    combined_df = pd.concat(all_candidates, ignore_index=True)
    combined_df = combined_df.sort_values(
        ["priority_score", "shot_count", "best_mean_abs_error_sec"],
        ascending=[False, False, True],
        kind="stable",
    ).reset_index(drop=True)
else:
    combined_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

if WRITE_COMBINED_CSV:
    combined_path = OUTPUT_DIR / f"all_films_{suffix}.csv"
    combined_df.to_csv(combined_path, index=False, encoding="utf-8-sig")
    print(f"Combined output: {combined_path.resolve()}")

print(f"Films scanned: {len(wide_df)}")
print(f"Candidate rows: {len(combined_df)}")
print(f"Per-film files written: {len(wide_df) if WRITE_PER_FILM_CSV else 0}")
display(combined_df.head(30))

Films: 100%|██████████| 454/454 [11:40<00:00,  1.54s/it]

Combined output: /Users/caoshuangyuan/Desktop/Workspace/DirtyPilot/metric_montage_multi_base_results/all_films_shot3_len1to90_tlr0p2.csv
Films scanned: 454
Candidate rows: 12014
Per-film files written: 454


,film,start_shot,end_shot,shot_count,start_sec,end_sec,duration_total_sec,tolerance_sec,all_matching_bases,matching_base_count,...,best_expected_sec,best_errors_sec,best_max_abs_error_sec,best_mean_abs_error_sec,best_max_multiple,best_mean_multiple,compact_candidate_flag,low_information_base_flag,priority_score,video_path
0,秋翁遇仙记,149,154,6,1350.120,1420.120,70.000,0.2,2.0;5.0;10.0,3,...,20.0;10.0;10.0;10.0;10.0;10.0,0.0;0.0;0.0;0.0;0.0;0.0,0.000,0.000,2,1.167,True,False,75.000,/Volumes/Extreme SSD/Video/Chinese1949-66/秋翁遇仙...
1,耕云播雨,229,234,6,654.609,672.611,18.002,0.2,2.0,1,...,2.0;4.0;4.0;2.0;4.0;2.0,0.0;0.001;0.001;0.0;0.001;0.0,0.001,0.001,2,1.500,True,False,74.965,/Volumes/Extreme SSD/Video/Chinese1949-66/耕云播雨...
2,三滴血,788,795,8,6151.536,6197.542,46.006,0.2,2.0,1,...,4.0;6.0;20.0;2.0;6.0;2.0;4.0;2.0,0.001;0.001;0.003;0.0;0.001;0.0;0.001;0.0,0.003,0.001,10,2.875,False,False,74.945,/Volumes/Extreme SSD/Video/Chinese1949-66/三滴血.mp4
3,三滴血,919,924,6,6926.724,6944.646,17.922,0.2,2.0,1,...,2.0;2.0;4.0;4.0;2.0;4.0,0.0;0.0;0.001;0.001;-0.08;0.001,0.080,0.014,2,1.500,True,False,73.850,/Volumes/Extreme SSD/Video/Chinese1949-66/三滴血.mp4
4,庵堂认母,10,15,6,101.013,113.015,12.002,0.2,2.0,1,...,2.0;2.0;2.0;2.0;2.0;2.0,0.04;-0.04;0.04;-0.04;0.04;-0.04,0.040,0.040,1,1.000,True,False,73.600,/Volumes/Extreme SSD/Video/Chinese1949-66/庵堂认母...
5,乘风破浪,42,48,7,370.000,408.280,38.280,0.2,1.0;2.0,2,...,2.0;8.0;4.0;10.0;6.0;4.0;4.0,0.08;0.12;0.16;-0.08;0.08;-0.12;0.04,0.160,0.097,5,2.714,False,False,70.975,/Volumes/Extreme SSD/Video/Chinese1949-66/乘风破浪...
6,红梅岭,1105,1110,6,4237.827,4279.633,41.806,0.2,3.0,1,...,6.0;6.0;6.0;9.0;3.0;12.0,-0.199;0.001;0.001;0.161;-0.16;0.002,0.199,0.087,4,2.333,True,False,70.835,/Volumes/Extreme SSD/Video/Chinese1949-66/红梅岭.mp4
7,南岛风云,76,82,7,1369.000,1428.960,59.960,0.2,1.0;2.0,2,...,6.0;10.0;6.0;10.0;16.0;10.0;2.0,-0.04;0.0;-0.04;0.0;0.12;0.08;-0.16,0.160,0.063,8,4.286,False,False,65.825,/Volumes/Extreme SSD/Video/Chinese1949-66/南岛风云...
8,女审,1094,1098,5,5314.313,5324.314,10.001,0.2,2.0,1,...,2.0;2.0;2.0;2.0;2.0,0.0;0.0;0.0;0.0;0.0,0.000,0.000,1,1.000,True,False,65.000,/Volumes/Extreme SSD/Video/Chinese1949-66/女审.mp4
9,一文钱,58,62,5,75.970,93.973,18.003,0.2,1.0;3.0,2,...,3.0;3.0;6.0;3.0;3.0,0.0;0.0;0.001;0.0;0.0,0.001,0.000,2,1.200,True,False,64.990,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4


In [8]:
# Sanity-check helpers after running the full candidate generation cell.
for film in ["一文钱", "浪涛滚滚"]:
    film_candidates = combined_df[combined_df["film"] == film].copy()
    print(f"\n{film}: {len(film_candidates)} candidate(s)")
    display(film_candidates.head(10))

print("\nTop compact candidates:")
display(
    combined_df[combined_df["compact_candidate_flag"] & ~combined_df["low_information_base_flag"]]
    .sort_values(["priority_score", "shot_count"], ascending=[False, False])
    .head(25)
)


一文钱: 144 candidate(s)


,film,start_shot,end_shot,shot_count,start_sec,end_sec,duration_total_sec,tolerance_sec,all_matching_bases,matching_base_count,...,best_expected_sec,best_errors_sec,best_max_abs_error_sec,best_mean_abs_error_sec,best_max_multiple,best_mean_multiple,compact_candidate_flag,low_information_base_flag,priority_score,video_path
9,一文钱,58,62,5,75.970,93.973,18.003,0.2,1.0;3.0,2,...,3.0;3.0;6.0;3.0;3.0,0.0;0.0;0.001;0.0;0.0,0.001,0.000,2,1.200,True,False,64.990,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
20,一文钱,872,876,5,1591.215,1609.138,17.923,0.2,2.0,1,...,4.0;2.0;2.0;8.0;2.0,0.001;0.0;0.0;-0.079;0.0,0.079,0.016,4,1.800,True,False,63.810,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
89,一文钱,1322,1325,4,2253.305,2265.266,11.961,0.2,2.0,1,...,2.0;6.0;2.0;2.0,-0.04;0.001;0.04;-0.04,0.040,0.030,3,1.500,True,False,53.850,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
197,一文钱,100,102,3,178.984,184.985,6.001,0.2,2.0,1,...,2.0;2.0;2.0,0.0;0.0;0.0,0.000,0.000,1,1.000,True,False,45.000,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
198,一文钱,1466,1468,3,2471.014,2477.015,6.001,0.2,2.0,1,...,2.0;2.0;2.0,0.0;0.0;0.0,0.000,0.000,1,1.000,True,False,45.000,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
199,一文钱,2220,2222,3,3666.656,3672.657,6.001,0.2,2.0,1,...,2.0;2.0;2.0,0.0;0.0;0.0,0.000,0.000,1,1.000,True,False,45.000,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
268,一文钱,1374,1376,3,2336.796,2344.797,8.001,0.2,2.0,1,...,2.0;4.0;2.0,0.0;0.001;0.0,0.001,0.000,2,1.333,True,False,44.990,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
346,一文钱,88,90,3,155.981,165.982,10.001,0.2,2.0,1,...,2.0;4.0;4.0,0.0;0.001;0.001,0.001,0.001,2,1.667,True,False,44.965,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
347,一文钱,410,412,3,842.394,852.395,10.001,0.2,2.0,1,...,2.0;4.0;4.0,0.0;0.001;0.001,0.001,0.001,2,1.667,True,False,44.965,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
405,一文钱,887,889,3,1624.140,1630.100,5.960,0.2,2.0,1,...,2.0;2.0;2.0,0.0;0.0;-0.04,0.040,0.013,1,1.000,True,False,44.275,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4



浪涛滚滚: 18 candidate(s)


,film,start_shot,end_shot,shot_count,start_sec,end_sec,duration_total_sec,tolerance_sec,all_matching_bases,matching_base_count,...,best_expected_sec,best_errors_sec,best_max_abs_error_sec,best_mean_abs_error_sec,best_max_multiple,best_mean_multiple,compact_candidate_flag,low_information_base_flag,priority_score,video_path
30,浪涛滚滚,346,350,5,5299.537,5332.862,33.325,0.2,1.0;3.0,2,...,3.0;12.0;6.0;6.0;6.0,0.08;0.122;0.041;0.041;0.041,0.122,0.065,4,2.200,True,False,62.155,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
118,浪涛滚滚,347,350,4,5302.617,5332.862,30.245,0.2,2.0;6.0,2,...,12.0;6.0;6.0;6.0,0.122;0.041;0.041;0.041,0.122,0.061,2,1.250,True,False,52.255,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
884,浪涛滚滚,183,185,3,2840.035,2857.637,17.602,0.2,3.0;6.0,2,...,6.0;6.0;6.0,-0.159;-0.199;-0.039,0.199,0.132,1,1.000,True,False,39.710,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
1845,浪涛滚滚,182,185,4,2817.952,2857.637,39.685,0.2,1.0;2.0,2,...,22.0;6.0;6.0;6.0,0.083;-0.159;-0.199;-0.039,0.199,0.120,11,5.000,False,False,28.010,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
2243,浪涛滚滚,81,83,3,1325.584,1363.310,37.726,0.2,2.0,1,...,14.0;16.0;8.0,-0.038;-0.198;-0.039,0.198,0.092,8,6.333,False,False,24.720,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
2329,浪涛滚滚,274,276,3,4310.439,4350.525,40.086,0.2,1.0;2.0,2,...,10.0;12.0;18.0,0.041;0.122;-0.078,0.122,0.080,9,6.667,False,False,23.780,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
8099,浪涛滚滚,461,463,3,6785.224,6800.306,15.082,0.2,1.0,1,...,3.0;2.0;10.0,0.0;0.0;0.081,0.081,0.027,10,5.000,False,True,-31.485,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
8945,浪涛滚滚,80,84,5,1302.741,1380.352,77.611,0.2,1.0,1,...,23.0;14.0;16.0;8.0;17.0,-0.157;-0.038;-0.198;-0.039;0.042,0.198,0.095,23,15.600,False,True,-40.355,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
9900,浪涛滚滚,403,405,3,6162.297,6204.143,41.846,0.2,1.0,1,...,22.0;15.0;5.0,-0.117;0.042;-0.079,0.117,0.079,22,14.000,False,True,-57.145,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...
10031,浪涛滚滚,41,43,3,666.933,699.977,33.044,0.2,1.0,1,...,23.0;8.0;2.0,-0.077;0.161;-0.04,0.161,0.093,23,11.000,False,True,-59.935,/Volumes/Extreme SSD/Video/Chinese1949-66/浪涛滚滚...



Top compact candidates:


,film,start_shot,end_shot,shot_count,start_sec,end_sec,duration_total_sec,tolerance_sec,all_matching_bases,matching_base_count,...,best_expected_sec,best_errors_sec,best_max_abs_error_sec,best_mean_abs_error_sec,best_max_multiple,best_mean_multiple,compact_candidate_flag,low_information_base_flag,priority_score,video_path
0,秋翁遇仙记,149,154,6,1350.120,1420.120,70.000,0.2,2.0;5.0;10.0,3,...,20.0;10.0;10.0;10.0;10.0;10.0,0.0;0.0;0.0;0.0;0.0;0.0,0.000,0.000,2,1.167,True,False,75.000,/Volumes/Extreme SSD/Video/Chinese1949-66/秋翁遇仙...
1,耕云播雨,229,234,6,654.609,672.611,18.002,0.2,2.0,1,...,2.0;4.0;4.0;2.0;4.0;2.0,0.0;0.001;0.001;0.0;0.001;0.0,0.001,0.001,2,1.500,True,False,74.965,/Volumes/Extreme SSD/Video/Chinese1949-66/耕云播雨...
3,三滴血,919,924,6,6926.724,6944.646,17.922,0.2,2.0,1,...,2.0;2.0;4.0;4.0;2.0;4.0,0.0;0.0;0.001;0.001;-0.08;0.001,0.080,0.014,2,1.500,True,False,73.850,/Volumes/Extreme SSD/Video/Chinese1949-66/三滴血.mp4
4,庵堂认母,10,15,6,101.013,113.015,12.002,0.2,2.0,1,...,2.0;2.0;2.0;2.0;2.0;2.0,0.04;-0.04;0.04;-0.04;0.04;-0.04,0.040,0.040,1,1.000,True,False,73.600,/Volumes/Extreme SSD/Video/Chinese1949-66/庵堂认母...
6,红梅岭,1105,1110,6,4237.827,4279.633,41.806,0.2,3.0,1,...,6.0;6.0;6.0;9.0;3.0;12.0,-0.199;0.001;0.001;0.161;-0.16;0.002,0.199,0.087,4,2.333,True,False,70.835,/Volumes/Extreme SSD/Video/Chinese1949-66/红梅岭.mp4
8,女审,1094,1098,5,5314.313,5324.314,10.001,0.2,2.0,1,...,2.0;2.0;2.0;2.0;2.0,0.0;0.0;0.0;0.0;0.0,0.000,0.000,1,1.000,True,False,65.000,/Volumes/Extreme SSD/Video/Chinese1949-66/女审.mp4
9,一文钱,58,62,5,75.970,93.973,18.003,0.2,1.0;3.0,2,...,3.0;3.0;6.0;3.0;3.0,0.0;0.0;0.001;0.0;0.0,0.001,0.000,2,1.200,True,False,64.990,/Volumes/Extreme SSD/Video/Chinese1949-66/一文钱.mp4
10,春风吹到诺敏河,1009,1013,5,2773.547,2793.550,20.003,0.2,2.0,1,...,2.0;6.0;8.0;2.0;2.0,0.0;0.001;0.001;0.0;0.0,0.001,0.000,4,2.000,True,False,64.990,/Volumes/Extreme SSD/Video/Chinese1949-66/春风吹到...
11,海上风暴,21,25,5,53.487,69.489,16.002,0.2,2.0,1,...,6.0;2.0;4.0;2.0;2.0,0.001;0.0;0.001;0.0;0.0,0.001,0.000,3,1.600,True,False,64.990,/Volumes/Extreme SSD/Video/Chinese1949-66/海上风暴...
12,秦娘美,280,284,5,1769.000,1783.002,14.002,0.2,2.0,1,...,6.0;2.0;2.0;2.0;2.0,0.001;0.0;0.0;0.0;0.0,0.001,0.000,3,1.400,True,False,64.990,/Volumes/Extreme SSD/Video/Chinese1949-66/秦娘美.mp4
